# 005 — BM25 Full-Text Retrieval
**Storage Series** | Sparse retrieval — the best complement to dense search

BM25 (Best Match 25) is the gold-standard sparse retrieval algorithm used by
Elasticsearch and Lucene. It excels at exact-keyword and rare-term queries where
dense embeddings often fail.


In [ ]:
!pip install -q rank_bm25

In [ ]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


In [ ]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


In [ ]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


In [ ]:
# ── Build BM25 index ─────────────────────────────────────────────────────
from rank_bm25 import BM25Okapi
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re, string

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
corpus = splitter.split_text(all_text[:20000])
print(f"Corpus size: {len(corpus)} chunks")

def tokenize(text: str):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.split()

tokenized_corpus = [tokenize(doc) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)
print("✓ BM25 index built")


In [ ]:
# ── BM25 retrieval ───────────────────────────────────────────────────────
def bm25_search(query: str, k: int = 5):
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [(corpus[i], scores[i]) for i in top_idx]

# Test queries
queries = [
    "Turing test artificial intelligence",
    "backpropagation gradient descent",
    "transformer attention mechanism",
]

for q in queries:
    results = bm25_search(q, k=2)
    print(f"Query: {q!r}")
    for text, score in results:
        print(f"  score={score:.3f}: {text[:120]}...")
    print()


In [ ]:
# ── LangChain BM25Retriever ───────────────────────────────────────────────
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

docs = [Document(page_content=c) for c in corpus]
bm25_retriever = BM25Retriever.from_documents(docs, k=4)

query = "machine learning supervised learning classification"
results = bm25_retriever.invoke(query)

print(f"BM25Retriever results for: {query!r}")
for i, r in enumerate(results):
    print(f"\n[{i+1}] {r.page_content[:250]}")


In [ ]:
# ── BM25 vs Dense: what each excels at ───────────────────────────────────
from langchain_community.vectorstores import FAISS
import pandas as pd

faiss_store     = FAISS.from_documents(docs, embeddings)
dense_retriever = faiss_store.as_retriever(search_kwargs={"k": 3})

test_cases = [
    ("Turing 1950 test for machine intelligence", "Keyword/exact match → BM25 wins"),
    ("What techniques let computers understand language?", "Semantic → Dense wins"),
    ("GPT-4 large language model capabilities",    "Named entity → BM25 wins"),
    ("How can AI systems reason about uncertainty?","Conceptual → Dense wins"),
]

for query, expected_winner in test_cases:
    bm25_top = bm25_retriever.invoke(query)[0].page_content[:100]
    dense_top = dense_retriever.invoke(query)[0].page_content[:100]
    print(f"Q: {query[:55]!r}")
    print(f"  Winner hint   : {expected_winner}")
    print(f"  BM25 top      : {bm25_top}...")
    print(f"  Dense top     : {dense_top}...")
    print()


## Key Takeaways
- BM25 is **term-frequency** based: rewards exact keyword matches
- Dense embeddings are **semantic**: rewards meaning-level similarity
- Neither alone is optimal — combine them (see **006_hybrid_retrieval_rrf.ipynb**)
- BM25 is especially valuable for: product names, error codes, dates, acronyms
